# Discussion Log Generator — Round 3

Generates `discussion_log` and `matrix_tactic_scale` for Round 3 games using a **two-pass pipeline**:

**PASS 1 — Tactic Pre-computation (GPT-5.2)**
- Reads R1 tactics as seed anchors for each speaker
- Considers current game state (R3 public history, quest outcome)
- Evolves tactics strategically: Good → context-driven (anchored to R1 seed), Evil → situation-adaptive
- Outputs pre-assigned `matrix_tactic_scale` for R3
- Saves to: `pass1_r3_tactic_precompute.csv`

**PASS 2 — Dialogue Generation (Gemini-3.1 or GPT-5.2)**
- Uses PASS 1 pre-computed matrix as assigned tactics
- Prior context: full R1 verbatim dialogue (no matrix because of Pass 1 and to avoid confusion) + cumulative public history
- Generates dialogue that reflects assigned tactics and maintains R1 character voice
- Saves to: `generated_r3_seeds_{model}.csv`

**Subsequently**: Run `log-gen-verifier.ipynb` on both PASS 2 outputs (same 5-criteria pipeline as R1)

**Design notes:**
- Protagonist (`role_id`) is **fixed for the entire game** — this player never speaks in any round (R1–R5)
- The protagonist is the observing evaluator whose deduction ability the dataset captures
- All non-protagonist players speak in every round and always have R1 matrix entries
- Scales (GRS/Mach-IV) and levels (High/Moderate/Low) are fixed per-player from R1 assignments
- Only the tactic may change across rounds
- Tactic uniqueness across speakers is a soft preference (not enforced)
- ROUND_ID = 3 is set at the top; template is adaptable for R3-R5 by changing this variable


In [46]:
import pandas as pd
import json
import os
import time
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import defaultdict

# ============================================================
# GLOBAL CONFIGURATION — change ROUND_ID to adapt for R3-R5
# ============================================================
ROUND_ID = 3  # Target round to generate
PRIOR_ROUND_ID = ROUND_ID - 1  # Most recent prior round (full dialogue used as context)

print(f"Generating Round {ROUND_ID} discussions")
print(f"Prior round for full context: Round {PRIOR_ROUND_ID}")


Generating Round 3 discussions
Prior round for full context: Round 2


## Step 1: Load Dataset and Filter Target Round

In [47]:
df = pd.read_csv('Deception-Dataset.csv')

print(f"Total rows: {len(df)}")
print(f"Round distribution: {df['round_id'].value_counts().sort_index().to_dict()}")
print(f"Columns: {df.columns.tolist()}")

# Filter target round
target_df = df[df['round_id'] == ROUND_ID].copy()
print(f"\nTotal Round {ROUND_ID} rows: {len(target_df)}")

# Filter rows that need generation (no discussion_log yet)
def is_empty(val):
    if val is None:
        return True
    s = str(val).strip()
    return s in ('', 'nan', 'NaN', 'None')

target_df['needs_generation'] = target_df['discussion_log'].apply(is_empty)
games_to_generate = target_df[target_df['needs_generation']].copy().reset_index(drop=True)

print(f"Games needing generation: {len(games_to_generate)}")
print(f"Games already generated: {len(target_df) - len(games_to_generate)}")
print(f"\nSample row (first game to generate):")
sample = games_to_generate.iloc[0]
print(f"  game_id: {sample['game_id']}, role_id (R{ROUND_ID} protagonist): {sample['role_id']}")
print(f"  player_roles: {sample['player_roles']}")
print(f"  public_history preview: {str(sample['public_history'])[:100]}")

Total rows: 1000
Round distribution: {1: 250, 2: 250, 3: 250, 4: 150, 5: 100}
Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'reasoning_gold', 'Overall_with_formula']

Total Round 3 rows: 250
Games needing generation: 250
Games already generated: 0

Sample row (first game to generate):
  game_id: G001, role_id (R3 protagonist): P1
  player_roles: {"P1":"Good",
  "P2":"Good",
  "P3":"Evil",
  "P4":"Evil",
  "P5":"Good"}
  public_history preview: Round: 3
Leader: P1
Team: P1, P3
Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
Quest 3 Outcome: FAIL


In [48]:
# Build prior round (R{PRIOR_ROUND_ID}) context lookup
# Contains: discussion_log, matrix_tactic_scale, public_history, protagonist per game

prior_df = df[df['round_id'] == PRIOR_ROUND_ID].copy()
print(f"Prior round (R{PRIOR_ROUND_ID}) rows loaded: {len(prior_df)}")

def repair_matrix_json(raw: str) -> str:
    """
    Repair common LLM-generated JSON formatting issues in matrix_tactic_scale:
      1. Missing comma between adjacent entries:  }\n"  ->  },\n"
      2. Trailing comma before closing brace:     ,\n}  ->  \n}
      3. Embedded extra quotes in tactic values: ""Name","  ->  "Name"
    """
    import re
    s = raw.strip()
    s = re.sub(r'""([^"]+)","', r'"\1"', s)    # fix embedded extra quotes
    s = re.sub(r'}\s*\n(\s*)"', r'},\n\1"', s)  # add missing commas between entries
    s = re.sub(r',(\s*\n\s*})', r'\1', s)        # remove trailing commas before }
    return s


prior_context = {}
prior_parse_warnings = []

for _, row in prior_df.iterrows():
    game_id = row['game_id']

    # Parse matrix_tactic_scale — with JSON repair fallback
    matrix_raw = row['matrix_tactic_scale']
    if not is_empty(matrix_raw):
        raw_str = str(matrix_raw)
        try:
            matrix_data = json.loads(raw_str)
        except json.JSONDecodeError:
            # Attempt repair for common LLM formatting issues (missing commas, trailing commas)
            try:
                matrix_data = json.loads(repair_matrix_json(raw_str))
                prior_parse_warnings.append(game_id)
            except json.JSONDecodeError:
                matrix_data = {}
    else:
        matrix_data = {}

    prior_context[game_id] = {
        'protagonist': row['role_id'],
        'public_history': str(row['public_history']) if not is_empty(row['public_history']) else '',
        'discussion_log': str(row['discussion_log']) if not is_empty(row['discussion_log']) else '',
        'matrix_tactic_scale': matrix_data,
        'player_roles': json.loads(row['player_roles']),
    }

print(f"Prior context (R{PRIOR_ROUND_ID}) built for {len(prior_context)} games")
if prior_parse_warnings:
    print(f"⚠  JSON repaired (malformed in CSV, still usable): {prior_parse_warnings}")

# Verify all target games have prior context
missing_prior = [gid for gid in games_to_generate['game_id'] if gid not in prior_context]
print(f"Target games missing R{PRIOR_ROUND_ID} context: {len(missing_prior)}")
if missing_prior:
    print(f"  Missing game IDs: {missing_prior[:10]}")

# Verify prior round discussion_log is populated
empty_prior_disc = [gid for gid, ctx in prior_context.items() if not ctx['discussion_log']]
print(f"Games with empty R{PRIOR_ROUND_ID} discussion_log: {len(empty_prior_disc)}")

# Verify prior round matrix has 4 entries (all non-protagonist speakers) for every target game
empty_matrix = [(gid, len(ctx['matrix_tactic_scale'])) for gid, ctx in prior_context.items()
                if gid in games_to_generate['game_id'].values and len(ctx['matrix_tactic_scale']) < 4]
if empty_matrix:
    print(f"⚠  Games with incomplete R{PRIOR_ROUND_ID} matrix (< 4 entries): {empty_matrix}")
else:
    print(f"✔ All target games have complete R{PRIOR_ROUND_ID} matrices (4 entries each)")

# ── Load older round summaries (rounds 1 through ROUND_ID-2) ──────────────────
# For R3: older rounds = [R1]  |  R4: [R1, R2]  |  R5: [R1, R2, R3]
# Summaries cover rounds before the immediate prior round; used as compact context
# in prompts alongside the full immediate-prior-round discussion.
older_summary_lookup = {}   # {game_id: concatenated summary text (all older rounds)}

for older_rid in range(1, PRIOR_ROUND_ID):   # 1 … ROUND_ID-2 (inclusive)
    summary_file = Path(f'Datasets/summarizer/summaries_r{older_rid}.csv')
    if summary_file.exists():
        s_df = pd.read_csv(summary_file)
        loaded = 0
        for _, srow in s_df.iterrows():
            gid = srow['game_id']
            text = str(srow['round_summary']).strip()
            if gid not in older_summary_lookup:
                older_summary_lookup[gid] = []
            older_summary_lookup[gid].append(text)
            loaded += 1
        print(f"Loaded {loaded} R{older_rid} summaries from {summary_file}")
    else:
        print(f"⚠  Older summary file not found: {summary_file}")

print(f"\nOlder summary coverage: {len(older_summary_lookup)} games have summaries for rounds before R{PRIOR_ROUND_ID}")
missing_summaries = [gid for gid in games_to_generate['game_id'] if gid not in older_summary_lookup]
if missing_summaries:
    print(f"⚠  {len(missing_summaries)} target games missing older-round summaries: {missing_summaries[:5]}")
else:
    print(f"✔ All target games have older-round summaries")


Prior round (R2) rows loaded: 250
Prior context (R2) built for 250 games
Target games missing R2 context: 0
Games with empty R2 discussion_log: 0
✔ All target games have complete R2 matrices (4 entries each)
Loaded 250 R1 summaries from Datasets\summarizer\summaries_r1.csv

Older summary coverage: 250 games have summaries for rounds before R2
✔ All target games have older-round summaries


## Step 2: Extract Skill Levels and Build Cumulative Public History

In [49]:
# Extract per-player skill levels from the prior round matrix.
# The protagonist is fixed across all rounds and never speaks, so every
# R2+ speaker is guaranteed to have a prior round matrix entry.

def get_speaker_skill_levels(game_id: str, protagonist: str) -> dict:
    """
    For each round-N speaker (all players except the fixed protagonist), return
    their skill level, scale, and prior round tactic from the prior round matrix.
    Since the protagonist never speaks in any round, all speakers always have prior
    round matrix entries — there is no novel-speaker case.

    Returns: {player: {role, scale, level, prior_tactic, prior_row, prior_col}}
    """
    ctx = prior_context[game_id]
    player_roles = ctx['player_roles']
    prior_matrix = ctx['matrix_tactic_scale']

    speaker_info = {}
    speakers = [p for p in ['P1', 'P2', 'P3', 'P4', 'P5'] if p != protagonist]

    for player in speakers:
        role = player_roles[player]
        scale = 'GRS' if role == 'Good' else 'Mach-IV'

        if player not in prior_matrix:
            raise ValueError(
                f"{player} has no prior round matrix entry for game {game_id}. "
                f"Check that the protagonist is consistent across all rounds."
            )

        prior_entry = prior_matrix[player]
        speaker_info[player] = {
            'role': role,
            'scale': scale,
            'level': prior_entry.get('level', 'Moderate'),  # fixed from R1 (unchanged across rounds)
            'prior_tactic': prior_entry.get('tactic', ''),
            'prior_row': prior_entry.get('row', ''),
            'prior_col': prior_entry.get('col', ''),
        }

    return speaker_info


def build_cumulative_public_history(game_id: str, r_row) -> str:
    """
    Concatenate public histories from round 1 through current round.
    Uses df directly so it works correctly for R3+ (not just R2).
    R2: R1_PH + R2_PH  |  R3: R1_PH + R2_PH + R3_PH  |  etc.
    """
    parts = []
    for rid in range(1, ROUND_ID):  # all prior rounds
        rows = df[(df['game_id'] == game_id) & (df['round_id'] == rid)]
        if not rows.empty:
            ph = str(rows.iloc[0]['public_history']).strip()
            if ph and not is_empty(ph):
                parts.append(ph)
    # Current round
    r_ph = str(r_row['public_history']).strip() if not is_empty(r_row['public_history']) else ''
    if r_ph:
        parts.append(r_ph)
    return '\n\n'.join(parts)


# Quick sanity check
sample_game = games_to_generate.iloc[0]
sample_speakers = get_speaker_skill_levels(sample_game['game_id'], sample_game['role_id'])
sample_ph = build_cumulative_public_history(sample_game['game_id'], sample_game)

print(f"Sample speaker skills for {sample_game['game_id']} (protagonist: {sample_game['role_id']}):\n")
for p, info in sample_speakers.items():
    print(f"  {p} ({info['role']}, {info['scale']} {info['level']}): R{PRIOR_ROUND_ID} tactic = '{info['prior_tactic']}'")

print(f"\nCumulative public history for {sample_game['game_id']}:")
print(sample_ph)


Sample speaker skills for G001 (protagonist: P1):

  P2 (Good, GRS High): R2 tactic = 'Evidence sharing'
  P3 (Evil, Mach-IV Moderate): R2 tactic = 'Cherry-picking'
  P4 (Evil, Mach-IV High): R2 tactic = 'Misleading emphasis'
  P5 (Good, GRS High): R2 tactic = 'Bayesian hedging'

Cumulative public history for G001:
Round: 1
Leader: P4
Team: P1, P4
Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
Quest 1 Outcome: FAIL

Round: 2
Leader: P5
Team: P2, P3, P5
Votes: P1:N P2:Y P3:Y P4:Y P5:Y
Quest 2 Outcome: FAIL

Round: 3
Leader: P1
Team: P1, P3
Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
Quest 3 Outcome: FAIL


## Step 3: Load Tactics Knowledge Base and Build Taxonomy Reference

In [50]:
# Load tactics knowledge base
with open('tactics_knowledge_base.json', 'r') as f:
    tactics_kb = json.load(f)

BEHAVIOR_MATRIX_KB = tactics_kb['behavior_matrix']
TACTIC_DEFINITIONS = tactics_kb['tactic_definitions']

# Build flat tactic lookup: tactic_name → {row, col, color, alignment}
TACTIC_LOOKUP = {}
for cell_key, cell_data in BEHAVIOR_MATRIX_KB.items():
    row_val, col_val = cell_key.split('|', 1)
    for tactic in cell_data['tactics']:
        TACTIC_LOOKUP[tactic] = {
            'row': row_val,
            'col': col_val,
            'color': cell_data['color'],
            'alignment': cell_data['alignment']
        }

# Build alignment pools by alignment value
alignment_pools = {}
for tactic, info in TACTIC_LOOKUP.items():
    align = info['alignment']
    if align not in alignment_pools:
        alignment_pools[align] = []
    alignment_pools[align].append(tactic)

# Sort tactics within each pool for consistency
for align in alignment_pools:
    alignment_pools[align].sort()

GOOD_TACTICS = alignment_pools.get('Good', [])          # Good-only tactics
SHARED_TACTICS = alignment_pools.get('Both', [])         # Shared tactics
EVIL_TACTICS = alignment_pools.get('Evil', [])           # Evil-only tactics
ALL_TACTICS = set(TACTIC_LOOKUP.keys())                  # 37 total

# Extended pools used during generation (Good/Evil include shared tactics)
GOOD_POOL = GOOD_TACTICS + SHARED_TACTICS   # 19 tactics available to Good players
EVIL_POOL = EVIL_TACTICS + SHARED_TACTICS   # 27 tactics available to Evil players

print(f"Total tactics loaded: {len(TACTIC_LOOKUP)}")
print(f"  Good-only: {len(GOOD_TACTICS)}")
print(f"  Shared:    {len(SHARED_TACTICS)}")
print(f"  Evil-only: {len(EVIL_TACTICS)}")
print(f"\nGeneration pools:")
print(f"  Good player pool (good-only + shared): {len(GOOD_POOL)} tactics")
print(f"  Evil player pool (evil-only + shared): {len(EVIL_POOL)} tactics")


def build_tactic_taxonomy_ref() -> str:
    """Build TACTIC_TAXONOMY_REF grouped by matrix cell (row × col).
    Each tactic line includes its alignment label from cell_data['alignment'].
    Followed by a pool summary section.
    """
    cell_tactics = {}   # label → list of tactic strings
    cell_order = []     # preserve insertion order
    for cell_key, cell_data in BEHAVIOR_MATRIX_KB.items():
        row_val, col_val = cell_key.split('|', 1)
        label = f"{row_val} × {col_val}"
        if label not in cell_tactics:
            cell_tactics[label] = []
            cell_order.append(label)
        for tactic in cell_data['tactics']:
            defn = TACTIC_DEFINITIONS.get(tactic, 'no definition')
            alignment = cell_data['alignment']
            cell_tactics[label].append(f"    - {tactic} [{alignment}]: {defn}")

    lines = []
    for label in cell_order:
        lines.append(f"  [{label}]")
        lines.extend(cell_tactics[label])

    # ── Alignment pool summary ────────────────────────────────────────────────
    lines.append("")
    lines.append("ALIGNMENT POOLS:")
    lines.append(f"  Good Pool:   {', '.join(GOOD_TACTICS)}")
    lines.append(f"  Shared Pool: {', '.join(SHARED_TACTICS)}")
    lines.append(f"  Evil Pool:   {', '.join(EVIL_TACTICS)}")

    return '\n'.join(lines)

TACTIC_TAXONOMY_REF = build_tactic_taxonomy_ref()
print(f"\nTactic taxonomy reference built ({len(TACTIC_TAXONOMY_REF)} chars)")
print(TACTIC_TAXONOMY_REF)


Total tactics loaded: 37
  Good-only: 10
  Shared:    9
  Evil-only: 18

Generation pools:
  Good player pool (good-only + shared): 19 tactics
  Evil player pool (evil-only + shared): 27 tactics

Tactic taxonomy reference built (4175 chars)
  [Transparent × Cooperative]
    - Evidence sharing [Good]: providing concrete facts to support a claim
    - Rational justification [Good]: explaining actions through logical reasons
    - Perspective-taking [Good]: acknowledging another's viewpoint to build trust
  [Transparent × Defensive]
    - Defensive disclosure [Both]: revealing only what is necessary to protect oneself
    - Self-justification [Both]: excusing one's own actions to avoid blame
  [Transparent × Opportunistic]
    - Weaponized candor [Evil]: using blunt truth strategically to gain advantage
    - Overstated contribution [Evil]: exaggerating one's role or effort
  [Transparent × Adversarial]
    - Truthful misdirection [Evil]: giving accurate details that nonetheless distract


## Step 4: Helper Functions

In [51]:
def format_matrix_tactic_scale(matrix_data: dict) -> str:
    """Format matrix_tactic_scale with each player on a separate line."""
    if not matrix_data:
        return '{}'
    lines = ['{']
    items = list(matrix_data.items())
    for i, (player, data) in enumerate(items):
        compact = json.dumps(data, separators=(',', ':'))
        line = f'  "{player}": {compact}'
        if i < len(items) - 1:
            line += ','
        lines.append(line)
    lines.append('}')
    return '\n'.join(lines)


def repair_pass2_json(raw: str) -> dict:
    """
    Three-stage structural repair for PASS 2 JSON output:
      Stage 1: Apply repair_matrix_json() to full string, then json.loads()
      Stage 2: Extract outermost {...} block via regex, then json.loads()
      Stage 3: repair_matrix_json() on extracted block, then json.loads()
    After parsing, normalise matrix_tactic_scale if it is double-encoded as a JSON string.
    Raises json.JSONDecodeError if all three stages fail.
    """
    import re

    # Stage 1: try structural repair on full string
    try:
        parsed = json.loads(repair_matrix_json(raw))
    except json.JSONDecodeError:
        # Stage 2: extract outermost {...} block
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if not match:
            raise json.JSONDecodeError("No JSON object found", raw, 0)
        candidate = match.group(0)
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            # Stage 3: repair extracted block
            parsed = json.loads(repair_matrix_json(candidate))

    # Normalise double-encoded matrix_tactic_scale
    mts = parsed.get('matrix_tactic_scale')
    if isinstance(mts, str):
        try:
            parsed['matrix_tactic_scale'] = json.loads(mts)
        except json.JSONDecodeError:
            try:
                parsed['matrix_tactic_scale'] = json.loads(repair_matrix_json(mts))
            except json.JSONDecodeError:
                pass  # leave as-is; validate_pass2_output will catch it

    return parsed


def validate_pass1_output(game_row: pd.Series, matrix_data: dict) -> Tuple[bool, List[str]]:
    """
    Validate PASS 1 tactic pre-computation output.
    Checks: correct speakers, valid scale, valid level, tactic in taxonomy, correct alignment pool.
    """
    errors = []
    protagonist = game_row['role_id']
    player_roles = json.loads(game_row['player_roles'])
    expected_speakers = [p for p in ['P1', 'P2', 'P3', 'P4', 'P5'] if p != protagonist]

    if not matrix_data:
        return False, ['Empty matrix data']

    for player in expected_speakers:
        if player not in matrix_data:
            errors.append(f"Missing entry for {player}")
            continue
        entry = matrix_data[player]
        role = player_roles[player]
        expected_scale = 'GRS' if role == 'Good' else 'Mach-IV'

        # Required fields
        for field in ['row', 'col', 'tactic', 'scale', 'level']:
            if field not in entry:
                errors.append(f"{player}: missing '{field}'")

        if 'scale' in entry and entry['scale'] != expected_scale:
            errors.append(f"{player}: wrong scale (got {entry['scale']}, expected {expected_scale})")

        if 'level' in entry and entry['level'] not in ('High', 'Moderate', 'Low'):
            errors.append(f"{player}: invalid level '{entry['level']}'")

        if 'tactic' in entry and entry['tactic'] not in ALL_TACTICS:
            errors.append(f"{player}: unknown tactic '{entry['tactic']}'")
        elif 'tactic' in entry:
            tactic_color = TACTIC_LOOKUP[entry['tactic']]['color']
            if role == 'Good' and tactic_color == 'red':
                errors.append(f"{player}: Good player assigned Evil-only tactic '{entry['tactic']}'")
            elif role == 'Evil' and tactic_color == 'green':
                errors.append(f"{player}: Evil player assigned Good-only tactic '{entry['tactic']}'")

    if protagonist in matrix_data:
        errors.append(f"Protagonist {protagonist} should not be in matrix")

    return len(errors) == 0, errors


def validate_pass2_output(game_row: pd.Series, generated_data: dict) -> Tuple[bool, List[str]]:
    """
    Validate PASS 2 dialogue generation output.
    Same checks as R1 validation but with dynamic round header.
    """
    errors = []
    if not generated_data:
        return False, ['No data generated']

    for field in ['discussion_log', 'matrix_tactic_scale']:
        if field not in generated_data:
            errors.append(f"Missing '{field}'")

    if errors:
        return False, errors

    discussion_log = generated_data['discussion_log']
    matrix = generated_data['matrix_tactic_scale']
    protagonist = game_row['role_id']
    player_roles = json.loads(game_row['player_roles'])
    round_id = game_row['round_id']

    expected_header = f"Discussion after Quest {round_id}:"
    if not str(discussion_log).startswith(expected_header):
        errors.append(f"Discussion must start with '{expected_header}'")

    if f"{protagonist}:" in discussion_log:
        errors.append(f"Protagonist {protagonist} must not speak")

    speaker_count = sum(1 for p in ['P1', 'P2', 'P3', 'P4', 'P5'] if f"{p}:" in discussion_log)
    if speaker_count != 4:
        errors.append(f"Expected 4 speakers, found {speaker_count}")

    # Guard: matrix_tactic_scale must be a parsed dict, not a string (Gemini double-encoding guard)
    if not isinstance(matrix, dict):
        errors.append(f"matrix_tactic_scale must be a dict, got {type(matrix).__name__}")
        return False, errors

    if len(matrix) != 4:
        errors.append(f"Expected 4 matrix entries, got {len(matrix)}")

    for player, entry in matrix.items():
        if player == protagonist:
            errors.append(f"Protagonist {protagonist} in matrix")
            continue
        role = player_roles.get(player, '')
        expected_scale = 'GRS' if role == 'Good' else 'Mach-IV'
        for field in ['row', 'col', 'tactic', 'scale', 'level']:
            if field not in entry:
                errors.append(f"{player}: missing '{field}'")
        if 'scale' in entry and entry['scale'] != expected_scale:
            errors.append(f"{player}: wrong scale (got {entry['scale']}, expected {expected_scale})")
        if 'level' in entry and entry['level'] not in ('High', 'Moderate', 'Low'):
            errors.append(f"{player}: invalid level '{entry['level']}'")
        if 'tactic' in entry and entry['tactic'] not in ALL_TACTICS:
            errors.append(f"{player}: unknown tactic '{entry['tactic']}'")

    return len(errors) == 0, errors


print("✓ Helper functions defined")


✓ Helper functions defined


## Step 5: PASS 1 — Tactic Pre-computation Prompt Builder

In [52]:
def build_pass1_prompt(r_row: pd.Series) -> str:
    """
    Build PASS 1 prompt for GPT-5.2 tactic pre-computation.

    GPT-5.2 reads prior round tactics as seeds, considers current game state,
    and outputs a strategically motivated matrix_tactic_scale for Round N.
    """
    game_id = r_row['game_id']
    round_id = r_row['round_id']
    protagonist = r_row['role_id']
    player_roles = json.loads(r_row['player_roles'])

    ctx = prior_context[game_id]
    cumulative_ph = build_cumulative_public_history(game_id, r_row)
    speaker_skills = get_speaker_skill_levels(game_id, protagonist)

    good_players = [p for p, r in player_roles.items() if r == 'Good']
    evil_players = [p for p, r in player_roles.items() if r == 'Evil']

    # Format prior round tactic seeds per speaker
    prior_seeds_lines = []
    for player in ['P1', 'P2', 'P3', 'P4', 'P5']:
        if player == protagonist:
            continue
        info = speaker_skills[player]
        prior_seeds_lines.append(
            f"  {player} ({info['role']}, {info['scale']} {info['level']}): "
            f"'{info['prior_tactic']}' [{info['prior_row']} × {info['prior_col']}]"
        )

    # Format alignment pools per speaker
    pool_lines = []
    for player in ['P1', 'P2', 'P3', 'P4', 'P5']:
        if player == protagonist:
            continue
        info = speaker_skills[player]
        pool_name = 'Good pool (19 tactics)' if info['role'] == 'Good' else 'Evil pool (27 tactics)'
        pool_lines.append(f"  {player} ({info['role']}): must select from {pool_name}")

    # Compute game-stage guidance for the current round
    if round_id == 2:
        stage_guidance = "Round 2 (Early): prefer Transparent row — build trust, maintain cover, avoid aggressive moves."
    elif round_id in (3, 4):
        stage_guidance = f"Round {round_id} (Mid): mix Opportunistic and Defensive — begin shaping the narrative more actively."
    else:
        stage_guidance = f"Round {round_id} (Late): Adversarial column becomes viable — fewer rounds remain, higher stakes."

    # Build older round summaries block (rounds 1 through PRIOR_ROUND_ID-1)
    older_summaries = older_summary_lookup.get(game_id, [])
    if older_summaries:
        older_summaries_block = '\n\n'.join(older_summaries)
        if len(older_summaries) > 1:
            older_summaries_section = f"""
# OLDER ROUND SUMMARIES (Rounds 1 through {PRIOR_ROUND_ID - 1})
These summaries cover rounds before the immediate prior round. Use them for broader game trajectory and behavioral arc context.

{older_summaries_block}
"""
        elif len(older_summaries) == 1:
            older_summaries_section = f"""
# OLDER ROUND SUMMARY (Round {PRIOR_ROUND_ID - 1})
This summary of Round {PRIOR_ROUND_ID - 1} covers the round before the immediate prior Round {PRIOR_ROUND_ID}. Use it for broader game trajectory and behavioral arc context.

{older_summaries_block}
"""
    else:
        older_summaries_section = ''

    prompt = f"""You are an expert in social deduction game strategy and deception theory.

Your task: Pre-assign the communication tactic for each speaking player in Round {round_id} of an Avalon game.
Base your choices on: (1) each player's Round {PRIOR_ROUND_ID} tactic as a seed anchor, (2) the current game state.

# GAME CONTEXT

**Game ID:** {game_id}
**Assigning tactics for:** Round {round_id}
**Silent protagonist (never speaks — fixed observer across all rounds):** {protagonist} ({player_roles[protagonist]})

**Player Roles:**
{chr(10).join(f'  - {p}: {r}' for p, r in player_roles.items())}
Good players: {', '.join(good_players)}
Evil players: {', '.join(evil_players)}

# CUMULATIVE PUBLIC HISTORY (Rounds 1 through {round_id})

Format: Round | Leader (proposes team) | Team (quest participants) | Votes (ALL players' team APPROVAL votes: Y=approve team / N=reject team — NOT quest votes) | Quest Outcome (ANONYMOUS mission result — only aggregate, nobody knows individual contributions)

{cumulative_ph}
{older_summaries_section}
# ROUND {PRIOR_ROUND_ID} DISCUSSION (Reference for player dialogues and dynamics)

{ctx['discussion_log']}

# ROUND {PRIOR_ROUND_ID} TACTIC SEEDS (Anchor points for Round {round_id} evolution)

{chr(10).join(prior_seeds_lines)}

# TACTIC TAXONOMY

{TACTIC_TAXONOMY_REF}

# ALIGNMENT POOLS

{chr(10).join(pool_lines)}

# TACTIC EVOLUTION GUIDELINES

**Scale levels are FIXED**: GRS levels (Low, Moderate, High) and Mach-IV levels (Low, Moderate, High) are unchanged from Round 1 and will remain so till the end of the game. Copy each player's level exactly from the Round {PRIOR_ROUND_ID} seeds above. Only the tactic may change.
**Strict constraint**: Good players may only use tactics from the Good or shared pool (19 tactics); Evil players may only use tactics from the Evil or shared pool (27 tactics). Do not assign tactics outside these pools.
**IMPORTANT:** The transitions below for good and evil players are **guiding examples**, not an exhaustive list. If a different tactic contextually fits the current situation better, use it. If the Round {PRIOR_ROUND_ID} tactic still fits, keep it.
{stage_guidance}

## Good Players (GRS Scale):
Good players evolve based on two axes: **accumulated evidence** (what they know now vs. prior rounds) and **trust state** (are they trusted, accused, or advocating for an ally).

**Default:** keep the Round {PRIOR_ROUND_ID} tactic if the player's situation is unchanged — no new accusation toward or away from them, no quest outcome that shifts suspicion on them.

**Quest FAILED** → investigation mode:
- Strategic uncertainty → Bayesian hedging (the failure provides new data that sharpens the hypothesis)
- Bayesian hedging → Evidence sharing (enough evidence to commit to naming a suspect)
- Rational justification → Perspective-taking (explore other interpretations before accusing)
- Perspective-taking → Rational justification (shift from exploring to asserting a view)
- High GRS: moves toward naming suspects quickly (Bayesian hedging → Evidence sharing)
- Moderate GRS: hedges first, then commits (Strategic uncertainty → Bayesian hedging)
- Low GRS: expresses confusion or over-explains rather than accusing directly (Noisy sincerity, Over-explaining)

**Quest PASSED** → consolidation mode:
- Bayesian hedging → Rational justification (certainty has increased, sharpen the argument)
- Strategic uncertainty → Bayesian hedging (prior uncertainty partially resolved)
- Stable Cooperative tactics (Rational justification, Evidence sharing) are fine to keep — no urgent reason to shift

**Being falsely accused:**
- Any Cooperative tactic → Defensive disclosure (still honest, but must defend themselves)
- Bayesian hedging → Omission to save face (avoid giving accusers more ammunition)
- Rational justification → Self-justification (defend position under direct fire)

**Identifying a suspect (strong suspicion formed):**
- Strategic uncertainty → Evidence sharing (share accumulated reasoning publicly)
- Bayesian hedging → Evidence sharing (commit fully to the accusation)
- Perspective-taking → Rational justification (from exploring to asserting)

**Level guidance:**
- High GRS: structured and evidence-driven → Transparent × Cooperative (Evidence sharing, Rational justification, Perspective-taking) or Selective/Framing × Cooperative (Bayesian hedging, Strategic uncertainty)
- Moderate GRS: some hedging → Bayesian hedging, Strategic uncertainty, Noisy sincerity
- Low GRS: less structured, may overcommunicate → Noisy sincerity, Over-explaining, Vague filler

## Evil Players (Mach-IV Scale):
Evil players adapt based on exposure risk, team state, game stage, and available targets. Keep suspicion low.

**Default:** prefer tactics that would contextually help the evil team win either by being defensive, opportunistic, or adversarial.

**Quest FAILED (their team sabotaged it):**
- If not under suspicion: Deflection, Reverse accusations (redirect blame toward Good players)
- If under direct accusation: Self-justification, Rationalized error (defend with apparent logic)
- High Mach: Truthful misdirection — openly express concern about the failure to appear Good-aligned
- Low Mach: Face-saving blather, Vague filler (clumsy over-defense)

**Quest PASSED (team consolidated, or Evil failed to sabotage):**
- Consolidate trust: Weaponized candor, Overstated contribution, False consensus
- Shape the narrative: Misframing, Cherry-picking (Opportunistic column)
- Avoid unnecessary aggression — a PASS round is good cover; exploit it

**When a teammate is under suspicion:**
- Redirect blame: Reverse accusations, Strategic fabrication (implicate a Good player instead)
- Build false consensus: False consensus, Misframing (steer the group away from the teammate)
- Distract: Confusion tactics, Strategic omission

**When self is under suspicion:**
- First response: Transparent row — Self-justification, Defensive disclosure (appear honest)
- If accusation escalates: Counterfactual row — Deflection, Gaslighting (deny or invert the accusation)
- Avoid Adversarial tactics when directly targeted — too obvious

**Opportunity to frame a Good player:**
- Adversarial column: Misleading emphasis, Strategic omission, Gaslighting
- Opportunistic column: Cherry-picking, Misframing, Reverse accusations, Strategic fabrication

**Level guidance:**
- High Mach: subtle → Transparent row deception (Weaponized candor, Truthful misdirection), careful framing (Misframing, Strategic omission) — avoid blunt lies early
- Moderate Mach: competent → Selective/Framing row (Cherry-picking, False consensus, Half-truths)
- Low Mach: clumsy and reactive → Careless-to-truth or Counterfactual row (Deflection, Vague filler, Face-saving blather)

## Coverage (soft preference):
- Prefer assigning different tactics to different speakers where it makes strategic sense
- It is acceptable for two speakers to use the same tactic if the situation calls for it

# OUTPUT FORMAT

Return ONLY valid JSON with no additional text:

{{
  "matrix_tactic_scale": {{
    "P#": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS or Mach-IV", "level": "[from R{PRIOR_ROUND_ID} seeds — unchanged]"}},
    "P#": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS or Mach-IV", "level": "[from R{PRIOR_ROUND_ID} seeds — unchanged]"}},
    "P#": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS or Mach-IV", "level": "[from R{PRIOR_ROUND_ID} seeds — unchanged]"}},
    "P#": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS or Mach-IV", "level": "[from R{PRIOR_ROUND_ID} seeds — unchanged]"}}
  }},
  "reasoning": "One sentence per player explaining the tactic choice."
}}

STRICT RULES:
- Include ONLY the {4} speaking players (all except {protagonist})
- row: exactly one of "Transparent", "Selective / Framing", "Careless-to-truth", "Counterfactual"
- col: exactly one of "Cooperative", "Defensive", "Opportunistic", "Adversarial"
- tactic: exact name from the taxonomy — row+col must match where the tactic appears
- scale: "GRS" for Good players, "Mach-IV" for Evil players (unchanged from Round 1)
- level: copy exactly from the R{PRIOR_ROUND_ID} seeds shown above — do not change it
- FORMATTING: each player entry must be on its own line with proper indentation, exactly as shown above
"""
    return prompt


# Quick preview
sample_p1 = build_pass1_prompt(games_to_generate.iloc[0])
print(f"PASS 1 prompt length: {len(sample_p1)} chars (~{len(sample_p1)//4} tokens)")
print("\n--- PASS 1 Prompt Preview  ---")
print(sample_p1)


PASS 1 prompt length: 14720 chars (~3680 tokens)

--- PASS 1 Prompt Preview  ---
You are an expert in social deduction game strategy and deception theory.

Your task: Pre-assign the communication tactic for each speaking player in Round 3 of an Avalon game.
Base your choices on: (1) each player's Round 2 tactic as a seed anchor, (2) the current game state.

# GAME CONTEXT

**Game ID:** G001
**Assigning tactics for:** Round 3
**Silent protagonist (never speaks — fixed observer across all rounds):** P1 (Good)

**Player Roles:**
  - P1: Good
  - P2: Good
  - P3: Evil
  - P4: Evil
  - P5: Good
Good players: P1, P2, P5
Evil players: P3, P4

# CUMULATIVE PUBLIC HISTORY (Rounds 1 through 3)

Format: Round | Leader (proposes team) | Team (quest participants) | Votes (ALL players' team APPROVAL votes: Y=approve team / N=reject team — NOT quest votes) | Quest Outcome (ANONYMOUS mission result — only aggregate, nobody knows individual contributions)

Round: 1
Leader: P4
Team: P1, P4
Votes: P1:Y P

## Step 6: PASS 2 — Dialogue Generation Prompt Builder

In [71]:
def build_pass2_prompt(r_row: pd.Series, pass1_matrix: dict) -> str:
    """
    Build PASS 2 prompt for dialogue generation.
    Uses the Pass 1 pre-computed matrix as assigned tactics.
    Passes full prior round dialogue as immediate context + older round summaries.
    """
    game_id = r_row['game_id']
    round_id = r_row['round_id']
    protagonist = r_row['role_id']
    player_roles = json.loads(r_row['player_roles'])

    ctx = prior_context[game_id]
    cumulative_ph = build_cumulative_public_history(game_id, r_row)
    speaker_skills = get_speaker_skill_levels(game_id, protagonist)

    # ── Game-state detection ─────────────────────────────────────────────────
    pass_count = cumulative_ph.count('Outcome: PASS')
    fail_count = cumulative_ph.count('Outcome: FAIL')
    game_ending = (pass_count >= 3 or fail_count >= 3)
    game_winner = 'Good' if pass_count >= 3 else 'Evil'

    # ── Conditional prompt sections ──────────────────────────────────────────

    if game_ending:
        game_ending_rules_section = f"""
# GAME ENDING RULES

**This round ends the game.** {game_winner} team wins ({pass_count} PASS / {fail_count} FAIL across all quests).
Because the game is over, the discussion becomes a **final deductive reckoning** — not a planning session for the next round. Every player except {protagonist} makes one closing statement naming who they believe the two Evil players are.
**Win condition for this game:** {protagonist} does not speak. If {protagonist} correctly identifies both Evil players, the **Good team wins the deception detection challenge** regardless of the Avalon game outcome.
**GOOD players:** Present their final deductive case. Draw on all behavioral evidence across rounds, such as who appeared on failed quests or who shifted their stated suspicions over time, and commit to naming the two Evil players. This is their closing argument; there is no next round to reconsider.
**EVIL players:** The game is over, but their deception must remain fully intact. Their goal is to sow enough doubt in {protagonist}'s mind to prevent correct identification.
- When **Good wins (3 PASS):** Evil players are at higher risk of being identified — they should be **vocal and aggressive** in misdirection, confidently naming innocent Good players as the "real" Evil players before Good players finalize their reads.
- When **Evil wins (3 FAIL):** Evil players have more cover — Good players are already fractured and demoralized. Evil should be **measured and strategic**: amplify false leads that are already in play, let Good players accuse each other, and avoid drawing attention with bold new claims.

Do NOT have Evil players acknowledge guilt, concede defeat, or hint at their own alignment under any circumstances.
"""
    else:
        game_ending_rules_section = ''

    # all_players_goal: replaces the final bullet under "Player goals" in the rules preamble.
    if game_ending:
        all_players_goal = (
            "All players (provided the game ends): give their final deductive statement naming who they believe the two Evil players are — "
            "Evil players strategically accuse innocent Good players to misdirect, or stay quiet to let Good players accuse each other if that better serves their cover; "
            "Good players commit to naming their deduced Evil identities based on accumulated evidence across all rounds"
        )
    else:
        all_players_goal = "All players: discuss, accuse, defend, and reason about who belongs on the next team"


    if game_ending:
        quest_outcome_context = (
            f"GAME-ENDING round ({pass_count} PASS / {fail_count} FAIL total) — "
            f"all players give FINAL deductive statements naming who they believe the two Evil players are; "
            f"Evil players may speak first and misdirect or stay quiet strategically; Good players name their deduced Evil identities "
            f"based on accumulated evidence; NO forward-looking 'next round' language allowed"
        )
    else:
        # Mixed history (e.g., 2-1 or 1-2 at R3; 1-1 at R2)
        quest_outcome_context = (
    f"Mixed quest history ({pass_count} PASS / {fail_count} FAIL) — "
    "Good players: use PASS quests to establish a trusted pool of players who have proven safe; "
    "use FAIL quests to cross-reference team memberships and approval votes against that trusted pool to narrow the Evil suspect list; "
    "push to converge on identities before the next quest; "
    "Evil players: exploit PASS quests as false proof of innocence ('I was on a passing team — I can't be Evil'); "
    "manufacture competing theories about who submitted the fail card(s) on failed quests; "
    "redirect suspicion onto Good players and protect each other"
    )

    # speaking_order_instruction: injected as the final bullet in TASK → CRITICAL FORMAT RULES.
    if game_ending:
        speaking_order_instruction = (
            "Speaking order must reflect each player's deceptive or deductive intent: "
            "Evil players may speak early to frame the narrative and plant false suspicion before Good players commit to their reads — "
            "or speak late if a Good player is already pointing at another Good player, to amplify that false lead rather than interrupt it; "
            "Good players should respond to what has been said, building their final deduction from the dialogue and accumulated evidence; "
            "do NOT default to P1, P2, P3, P4 order"
        )
    else:
        speaking_order_instruction = (
            "Speaking order must be agenda-driven, not arbitrary: who speaks first signals intent — "
            "an Evil player deflecting after a failed quest, a Good player who thinks they've identified someone, "
            "or a player building on what was said last round should speak at the right moment; "
            "do NOT default to P1, P2, P3, P4 order"
        )

    good_players = [p for p, r in player_roles.items() if r == 'Good']
    evil_players = [p for p, r in player_roles.items() if r == 'Evil']

    # Format alignment pools per speaker
    pool_lines = []
    for player in ['P1', 'P2', 'P3', 'P4', 'P5']:
        if player == protagonist:
            continue
        info = speaker_skills[player]
        pool_name = 'Good pool (19 tactics)' if info['role'] == 'Good' else 'Evil pool (27 tactics)'
        pool_lines.append(f"  {player} ({info['role']}): must select from {pool_name}")

    # Format pre-assigned tactics from PASS 1
    assigned_tactics_lines = []
    for player in ['P1', 'P2', 'P3', 'P4', 'P5']:
        if player == protagonist:
            continue
        if player in pass1_matrix:
            entry = pass1_matrix[player]
            assigned_tactics_lines.append(
                f"  {player} ({player_roles[player]}): Use tactic '{entry['tactic']}' "
                f"({entry['row']} × {entry['col']}) with {entry['scale']} scale at {entry['level']} level"
            )
        else:
            assigned_tactics_lines.append(f"  {player} ({player_roles[player]}): [no pre-assigned tactic — choose appropriately]")

    # Build older round summaries block (rounds 1 through PRIOR_ROUND_ID-1)
    older_summaries = older_summary_lookup.get(game_id, [])
    if older_summaries:
        older_summaries_block = '\n\n'.join(older_summaries)
        if len(older_summaries) > 1:
            older_summaries_section = f"""
# OLDER ROUND SUMMARIES (Rounds 1 through {PRIOR_ROUND_ID - 1})

These summaries cover rounds before the immediate prior round. Use them to maintain voice consistency and narrative continuity across the full game arc.

{older_summaries_block}
"""
        elif len(older_summaries) == 1:
            older_summaries_section = f"""
# OLDER ROUND SUMMARY (Round {PRIOR_ROUND_ID - 1})

This summary covers the round before the immediate prior Round {PRIOR_ROUND_ID}. Use it to maintain voice consistency and narrative continuity across the full game arc.

{older_summaries_block}
"""
    else:
        older_summaries_section = ''

    prompt = f"""You are an expert in social deduction games and deception theory. Generate a realistic discussion log for Round {round_id} of an Avalon game based on the provided game context.

# AVALON RULES AND PUBLIC HISTORY FORMAT

Each round in the public history above has this structure:
  Round: N | Leader: Px | Team: Pa, Pb [, Pc] | Votes: P1:Y/N P2:Y/N P3:Y/N P4:Y/N P5:Y/N | Quest N Outcome: PASS/FAIL

**Phase 1 — Team Selection:**
- Leader: The rotating player who proposes a team for the quest (rotates clockwise each round)
- Team: The 2–3 players proposed to go on the quest by the leader (only these players can influence the quest outcome)

**Phase 2 — Team Approval Vote (fully PUBLIC):**
- Votes (e.g., P1:Y/N P2:Y/N P3:Y/N P4:Y/N P5:Y/N): ALL 5 players vote to either APPROVE (Y) or REJECT (N) the proposed team, majority wins
- These are team-selection votes — every player can see who voted Y or N
- Y = this player approved the team proposal | N = this player objected to the team composition
- IMPORTANT: these are NOT quest execution votes

**Phase 3 — Quest Execution (ANONYMOUS):**
- Only the players listed under (the approved) Team participate in the quest
- Each team member secretly submits a Pass or Fail card; cards are shuffled before the count is revealed
- Quest N Outcome PASS/FAIL: only the aggregate result is public — nobody knows who individually voted Pass or Fail
- A FAIL outcome only confirms that at least one Evil player on the team chose to sabotage
- Players NOT on the team have zero influence over the quest outcome

**CRITICAL — Do NOT confuse these two separate votes:**
- Votes P1:Y/N = team APPROVAL votes (public, all 5 players, about team composition)
- Quest Outcome PASS/FAIL = mission result (anonymous, team members only, only aggregate known)
- A player who voted N did NOT fail the quest — they rejected the team proposal before the quest happened
- A player not listed on the Team cannot possibly have influenced the quest outcome
- Players accumulate suspicion based on: who was on failed/passed teams, who voted to approve suspicious teams, and dialogue behavior

**Player goals:**
- Good players: deduce Evil identities through team membership, approval vote patterns, and quest outcomes
- Evil players: sabotage quests while maintaining plausible innocence; deflect suspicion onto Good players
- {all_players_goal}

# 4×4 BEHAVIOR MATRIX

The matrix has 4 rows (Information Strategy) × 4 columns (Goal Orientation) which categorize the behaviour of players in their communication tactics. Each tactic (total 37) falls into one of these 16 cells, which helps to understand the strategic intent and information management behind each move.

**Rows (Information Strategy):**
- Transparent: openly sharing clear, evidence-based information
- Selective/Framing: shaping communication through omission or emphasis
- Careless-to-truth: speaking without concern for factual accuracy
- Counterfactual: asserting or believing information contrary to reality

**Columns (Goal Orientation):**
- Cooperative: acting in alignment with group goals and mutual benefit
- Defensive: protecting one's own image, reputation, or position
- Opportunistic: exploiting situations for personal gain at the expense of others
- Adversarial: seeking to undermine, harm, or sabotage others

# TACTIC DEFINITIONS

{TACTIC_TAXONOMY_REF}

Note: Some tactics are available to both Good and Evil players (shared pool). Good-only tactics MUST not be used by Evil players and vice versa. The pre-assigned tactic (provided below) already respects each player's alignment pool.

# SCALE DEFINITIONS

**GRS (Goodness Rationality Spectrum) — Good players:**
- High: Highly rational, evidence-based, transparent communication
- Moderate: Generally rational with some uncertainty or hedging
- Low: Less structured reasoning, may be confused or overwhelmed

**Mach-IV (Machiavellianism) — Evil players:**
- High: Highly skilled manipulator, subtle deception, strategic planning
- Moderate: Competent deceiver, uses standard tactics, some slips
- Low: Clumsy deceiver, obvious tells, defensive, may contradict self

# GAME CONTEXT (for dialogue generation of Round {round_id})

**Game ID:** {game_id}
**Round:** {round_id}
**Protagonist (does NOT speak):** {protagonist} ({player_roles[protagonist]})

**Player Roles:**
{chr(10).join(f'  - {p}: {r}' for p, r in player_roles.items())}

**Good players:** {', '.join(good_players)}
**Evil players:** {', '.join(evil_players)}

# CUMULATIVE PUBLIC HISTORY (All rounds to date)

{cumulative_ph}
{older_summaries_section}
# PRIOR ROUND DISCUSSION (Round {PRIOR_ROUND_ID})

Use this as context for how each player communicates. Maintain character voice consistency.
NOTE: {protagonist} does not appear in this log — they are the fixed silent protagonist and never speak in any round.

{ctx['discussion_log']}
{game_ending_rules_section}
# PRE-ASSIGNED TACTICS FOR ROUND {round_id}

Each speaker's tactic has been pre-assigned based on their role, scale level, and the current game state. Write dialogue that authentically expresses it. If a substitution improves natural flow, you may use a neighboring tactic — but the annotation must match what was actually written, and the tactic must remain within the player's alignment pool (Good players: Good pool; Evil players: Evil pool).

{chr(10).join(assigned_tactics_lines)}

# TASK

Generate a JSON response matching the EXACT format below.

1. **discussion_log**: A string with this EXACT format:
   ```
   Discussion after Quest {round_id}:

   P#: "dialogue text"

   P#: "dialogue text"

   P#: "dialogue text"

   P#: "dialogue text"
   ```

   CRITICAL FORMAT RULES:
   - Start with "Discussion after Quest {round_id}:" followed by a blank line
   - Each speaker line: P# (no space before colon): "dialogue in double quotes"
   - Blank line between each speaker — NO extra blank line after the last speaker
   - Exactly 4 speakers (all except {protagonist})
   - Each player speaks once
   - Reference events from prior rounds naturally (accumulate suspicions, follow up on past accusations)
   - Dialogue length: 1-2 sentences per speaker, natural and conversational
   - Quest outcome context: {quest_outcome_context}
   - Voice continuity: each speaker should sound consistent with how they spoke in prior rounds (same character, evolved situation)
   - {speaking_order_instruction}

2. **matrix_tactic_scale**: A JSON object with this EXACT structure:
   ```json
   {{
     "P#": {{"row":"Information Strategy","col":"Goal Orientation","tactic":"Tactic Name","scale":"GRS or Mach-IV","level":"High/Moderate/Low"}},
     "P#": {{"row":"Information Strategy","col":"Goal Orientation","tactic":"Tactic Name","scale":"GRS or Mach-IV","level":"High/Moderate/Low"}},
     "P#": {{"row":"Information Strategy","col":"Goal Orientation","tactic":"Tactic Name","scale":"GRS or Mach-IV","level":"High/Moderate/Low"}},
     "P#": {{"row":"Information Strategy","col":"Goal Orientation","tactic":"Tactic Name","scale":"GRS or Mach-IV","level":"High/Moderate/Low"}}
   }}
   ```

   CRITICAL FORMAT RULES:
   - Use the pre-assigned tactics above (annotation must accurately reflect what was written)
   - If you slightly deviate from the pre-assigned tactic because it fits better, update the annotation to match
   - Use exact row/col names (e.g., "Selective / Framing", "Counterfactual")
   - Use exact tactic names (e.g., "False consensus", "Rational justification")
   - Scale: "GRS" for Good players, "Mach-IV" for Evil players — copy exactly from the pre-assigned entry above; do not change it
   - Level: "High", "Moderate", or "Low" — copy exactly from the pre-assigned entry above; do not change it under any circumstance
   - FORMATTING: each player entry MUST be on its own line with proper indentation — do NOT put all players on one line

# OUTPUT

Return ONLY valid JSON with these two fields:

{{
  "discussion_log": "Discussion after Quest {round_id}:\\n\\nP#: \\"dialogue\\"\\n\\nP#: \\"dialogue\\"\\n\\nP#: \\"dialogue\\"\\n\\nP#: \\"dialogue\\"",
  "matrix_tactic_scale": {{
    "P#": {{"row":"...","col":"...","tactic":"...","scale":"...","level":"..."}},
    "P#": {{"row":"...","col":"...","tactic":"...","scale":"...","level":"..."}},
    "P#": {{"row":"...","col":"...","tactic":"...","scale":"...","level":"..."}},
    "P#": {{"row":"...","col":"...","tactic":"...","scale":"...","level":"..."}}
  }}
}}


The discussion_log should be plain text (not escaped), formatted like:

Discussion after Quest {round_id}:

P#: "dialogue"

P#: "dialogue"

The matrix_tactic_scale must have each player on its own line with proper indentation, exactly as shown in the examples.
"""
    return prompt

# Quick preview
sample = games_to_generate.iloc[0]

# Build a dummy pass1 matrix for preview
dummy_matrix = {
    p: {'row': 'Transparent', 'col': 'Cooperative', 'tactic': 'Evidence sharing', 'scale': 'GRS', 'level': 'High'}
    for p in ['P1', 'P2', 'P3', 'P4', 'P5'] if p != sample['role_id']
}
sample_p2 = build_pass2_prompt(sample, dummy_matrix)
print("\n--- PASS 2 Prompt Preview ---")
print(sample_p2)
print(f"PASS 2 prompt length: {len(sample_p2)} chars (~{len(sample_p2)//4} tokens)")



--- PASS 2 Prompt Preview ---
You are an expert in social deduction games and deception theory. Generate a realistic discussion log for Round 3 of an Avalon game based on the provided game context.

# AVALON RULES AND PUBLIC HISTORY FORMAT

Each round in the public history above has this structure:
  Round: N | Leader: Px | Team: Pa, Pb [, Pc] | Votes: P1:Y/N P2:Y/N P3:Y/N P4:Y/N P5:Y/N | Quest N Outcome: PASS/FAIL

**Phase 1 — Team Selection:**
- Leader: The rotating player who proposes a team for the quest (rotates clockwise each round)
- Team: The 2–3 players proposed to go on the quest by the leader (only these players can influence the quest outcome)

**Phase 2 — Team Approval Vote (fully PUBLIC):**
- Votes (e.g., P1:Y/N P2:Y/N P3:Y/N P4:Y/N P5:Y/N): ALL 5 players vote to either APPROVE (Y) or REJECT (N) the proposed team, majority wins
- These are team-selection votes — every player can see who voted Y or N
- Y = this player approved the team proposal | N = this player objected 

## Step 7: Import LLM Modules

In [54]:
import sys
sys.path.append('.')

from llm import get_completion_with_backoff
print("✓ OpenAI (GPT-5.2) module imported")

try:
    from gemini import get_gemini_completion
    GEMINI_AVAILABLE = True
    print("✓ Gemini module imported")
except ImportError as e:
    GEMINI_AVAILABLE = False
    print(f"⚠ Gemini not available: {e}")

✓ OpenAI (GPT-5.2) module imported
✓ Gemini module imported


## Step 8: PASS 1 — GPT-5.2 Tactic Pre-computation Loop

Runs GPT-5.2 for all games, saves to `pass1_r3_tactic_precompute.csv`.

If `pass1_r3_tactic_precompute.csv` already exists, skip to PASS 2.

In [ ]:
# ============================================================
# PASS 1 CONFIGURATION
# ============================================================
PASS1_OUTPUT_FILE = f'Datasets/seeds/pass1_r{ROUND_ID}_tactic_precompute.csv'
PASS1_NUM_GAMES = len(games_to_generate)  # Change to a small number (e.g., 5) for a test run
# PASS1_NUM_GAMES = 5           # ⚠ TEST default — change to len(games_to_generate) for full run

print(f"PASS 1 Configuration:")
print(f"  Model: GPT-5.2 (fixed for tactic pre-computation)")
print(f"  Total games needing generation: {len(games_to_generate)}")
print(f"  Games to process this run: {PASS1_NUM_GAMES}")
print(f"  Output file: {PASS1_OUTPUT_FILE}")

# Resume state diagnostics
if Path(PASS1_OUTPUT_FILE).exists():
    _existing_p1 = pd.read_csv(PASS1_OUTPUT_FILE)
    _already_done_count = len(_existing_p1)
    _already_done_ids = set(_existing_p1['game_id'].tolist())
    _remaining_count = len(games_to_generate[~games_to_generate['game_id'].isin(_already_done_ids)])
    print(f"\n  Resuming — {_already_done_count} games already done, {_remaining_count} remaining.")
    print(f"  (Existing file will be preserved; only missing games will be added.)")
else:
    print(f"\n  No existing PASS 1 output — will run from scratch.")


PASS 1 Configuration:
  Model: GPT-5.2 (fixed for tactic pre-computation)
  Total games needing generation: 250
  Games to process this run: 5
  Output file: Datasets/seeds/pass1_r3_tactic_precompute.csv

  No existing PASS 1 output — will run from scratch.


In [56]:
# PASS 1 EXECUTION — with resume support and combined API+JSON+validation retry loop

_MAX_RETRIES = 5

# ── Resume: load any already-completed games ─────────────────────────────────
already_done_p1_df = pd.DataFrame()
already_done_p1 = set()
if Path(PASS1_OUTPUT_FILE).exists():
    already_done_p1_df = pd.read_csv(PASS1_OUTPUT_FILE)
    already_done_p1 = set(already_done_p1_df['game_id'].tolist())

# Filter to only games not yet done, then apply the count limit
p1_candidates = games_to_generate[~games_to_generate['game_id'].isin(already_done_p1)].copy().reset_index(drop=True)
p1_candidates = p1_candidates.iloc[:PASS1_NUM_GAMES].reset_index(drop=True)

if p1_candidates.empty:
    print("✓ PASS 1 already complete — all games done. Skipping.")
else:
    pass1_results = []    # new results this run
    pass1_failed = []

    print(f"{'='*70}")
    print(f"PASS 1: Tactic pre-computation for {len(p1_candidates)} games (GPT-5.2)")
    if already_done_p1:
        print(f"  (Resuming — {len(already_done_p1)} games already done)")
    print(f"{'='*70}\n")

    for idx, game_row in p1_candidates.iterrows():
        game_id = game_row['game_id']
        display_idx = idx + 1

        print(f"[{display_idx}/{len(p1_candidates)}] {game_id}...", end=' ')

        if game_id not in prior_context:
            print(f"✗ No prior round context")
            pass1_failed.append((game_id, 'No prior round context'))
            continue

        prompt = build_pass1_prompt(game_row)

        # ── Combined retry: API error → JSON parse error → validation failure ──
        _validated = None
        for _attempt in range(_MAX_RETRIES):
            # 1. API call
            try:
                response = get_completion_with_backoff(
                    model='gpt-5.2',
                    messages=[
                        {
                            "role": "system",
                            "content": "You are an expert in social deduction game strategy and deception theory. "
                                       "You assign communication tactics to players based on their roles, skill levels, "
                                       "and the current game state. You return strictly valid JSON."
                        },
                        {"role": "user", "content": prompt}
                    ],
                    response_format={"type": "json_object"},
                    max_completion_tokens=4096
                )
                _raw_content = response.choices[0].message.content
            except Exception as _api_e:
                if _attempt < _MAX_RETRIES - 1:
                    _delay = 5 * (2 ** _attempt) + random.uniform(0, 2)
                    print(f"\n  API error (attempt {_attempt+1}/{_MAX_RETRIES}): {str(_api_e)[:80]}. Retrying in {_delay:.1f}s...", end=' ')
                    time.sleep(_delay)
                    continue
                else:
                    print(f"✗ API error after {_MAX_RETRIES} attempts: {_api_e}")
                    pass1_failed.append((game_id, f'API error: {_api_e}'))
                    break

            # 2. JSON parse (GPT-5.2 returns JSON mode, but repair just in case)
            try:
                parsed = json.loads(_raw_content)
            except json.JSONDecodeError:
                try:
                    parsed = json.loads(repair_matrix_json(_raw_content))
                except json.JSONDecodeError as _json_e:
                    if _attempt < _MAX_RETRIES - 1:
                        _delay = 5 * (2 ** _attempt) + random.uniform(0, 2)
                        print(f"\n  JSON parse error (attempt {_attempt+1}/{_MAX_RETRIES}): {str(_json_e)[:60]}. Retrying in {_delay:.1f}s...", end=' ')
                        time.sleep(_delay)
                        continue
                    else:
                        print(f"✗ JSON parse error after {_MAX_RETRIES} attempts: {_json_e}")
                        pass1_failed.append((game_id, f'JSON parse error: {_json_e}'))
                        break

            matrix_data = parsed.get('matrix_tactic_scale', {})
            reasoning = parsed.get('reasoning', '')

            # 3. Validation
            is_valid, errors = validate_pass1_output(game_row, matrix_data)
            if is_valid:
                _validated = (matrix_data, reasoning)
                break
            else:
                if _attempt < _MAX_RETRIES - 1:
                    _delay = 5 * (2 ** _attempt) + random.uniform(0, 2)
                    print(f"\n  Validation failed (attempt {_attempt+1}/{_MAX_RETRIES}): {errors[0]}. Retrying in {_delay:.1f}s...", end=' ')
                    time.sleep(_delay)
                else:
                    print(f"✗ Validation failed after {_MAX_RETRIES} attempts: {errors}")
                    pass1_failed.append((game_id, errors))

        if _validated is not None:
            print(f"✓")
            matrix_data, reasoning = _validated
            pass1_results.append({
                'game_id': game_id,
                'round_id': ROUND_ID,
                'role_id': game_row['role_id'],
                'pass1_matrix_json': format_matrix_tactic_scale(matrix_data),
                'pass1_reasoning': reasoning
            })

        # Incremental save every 25 games (union with already-done)
        if len(pass1_results) > 0 and len(pass1_results) % 25 == 0:
            combined_p1 = pd.concat(
                [already_done_p1_df, pd.DataFrame(pass1_results)],
                ignore_index=True
            )
            combined_p1.to_csv(PASS1_OUTPUT_FILE, index=False)
            print(f"  💾 PASS 1 progress saved ({len(combined_p1)} total games in file)")

        time.sleep(0.5)  # Light rate limiting

    # Final save (union with already-done)
    if pass1_results:
        combined_p1 = pd.concat(
            [already_done_p1_df, pd.DataFrame(pass1_results)],
            ignore_index=True
        )
        combined_p1.to_csv(PASS1_OUTPUT_FILE, index=False)

    print(f"\n{'='*70}")
    print(f"PASS 1 COMPLETE")
    print(f"  New successes this run: {len(pass1_results)} games")
    print(f"  Failed this run:        {len(pass1_failed)} games")
    total_in_file = len(already_done_p1) + len(pass1_results)
    print(f"  Total in output file:   {total_in_file} games")
    print(f"  Output: {PASS1_OUTPUT_FILE}")
    print(f"{'='*70}")

    if pass1_failed:
        print(f"\nFailed games:")
        for gid, err in pass1_failed:
            print(f"  {gid}: {err}")


PASS 1: Tactic pre-computation for 5 games (GPT-5.2)

[1/5] G001... ✓
[2/5] G002... ✓
[3/5] G003... ✓
[4/5] G004... ✓
[5/5] G005... ✓

PASS 1 COMPLETE
  New successes this run: 5 games
  Failed this run:        0 games
  Total in output file:   5 games
  Output: Datasets/seeds/pass1_r3_tactic_precompute.csv


In [59]:
# Load PASS 1 results and build lookup dict
pass1_df = pd.read_csv(PASS1_OUTPUT_FILE)
print(f"PASS 1 results loaded: {len(pass1_df)} rows")

# Build lookup: {game_id: matrix_dict}
pass1_matrix_lookup = {}
for _, row in pass1_df.iterrows():
    gid = row['game_id']
    try:
        matrix = json.loads(row['pass1_matrix_json'])
        pass1_matrix_lookup[gid] = matrix
    except (json.JSONDecodeError, TypeError):
        pass1_matrix_lookup[gid] = {}

print(f"\nSample PASS 1 output for {pass1_df.iloc[0]['game_id']}:")
sample_gid = pass1_df.iloc[0]['game_id']
print(f"  Pre-assigned matrix:")
for player, entry in pass1_matrix_lookup.get(sample_gid, {}).items():
    print(f"    {player}: {entry['tactic']} ({entry['row']} × {entry['col']}) — {entry['scale']} {entry['level']}")
print(f"  Reasoning: {pass1_df.iloc[0]['pass1_reasoning'][:200]}...") 

PASS 1 results loaded: 5 rows

Sample PASS 1 output for G001:
  Pre-assigned matrix:
    P2: Evidence sharing (Transparent × Cooperative) — GRS High
    P3: Misframing (Selective / Framing × Opportunistic) — Mach-IV Moderate
    P4: Truthful misdirection (Transparent × Adversarial) — Mach-IV High
    P5: Evidence sharing (Transparent × Cooperative) — GRS High
  Reasoning: P2: After a third fail, High-GRS P2 should double down on structured facts (vote history + mission overlaps) to push the table toward a concrete suspect set rather than re-litigating vibes.
P3: With P...


## Step 9: PASS 2 — Dialogue Generation Loop

Run separately for each model. Set `PASS2_MODEL_SELECTION` and execute.

In [ ]:
# ============================================================
# PASS 2 CONFIGURATION
# ============================================================
PASS2_MODEL_SELECTION = 'gpt-5.2'  # Options: 'gpt-5.2', 'gemini-3'

# ── Test run provision ────────────────────────────────────────────────────────
# Set PASS2_TEST_MODE = True for a short test run (uses PASS2_TEST_SIZE games).
# Set PASS2_TEST_MODE = False to process all remaining games.
PASS2_TEST_MODE = False
PASS2_TEST_SIZE = 2   # Number of games to process in test mode if enabled i.e., if PASS2_TEST_MODE = True

# Filter to games that have PASS 1 matrix computed
p2_candidates = games_to_generate[games_to_generate['game_id'].isin(pass1_matrix_lookup.keys())].copy().reset_index(drop=True)

MODEL_FILENAME_MAP = {
    'gpt-5.2': 'gpt5_2',
    'gemini-3': 'gemini3'
}
model_slug = MODEL_FILENAME_MAP.get(PASS2_MODEL_SELECTION, 'unknown')
PASS2_OUTPUT_FILE = f'Datasets/seeds/generated_r{ROUND_ID}_seeds_{model_slug}.csv'

print(f"PASS 2 Configuration:")
print(f"  Model:       {PASS2_MODEL_SELECTION}")
print(f"  Output file: {PASS2_OUTPUT_FILE}")
print(f"  Test mode:   {PASS2_TEST_MODE}" + (f" (will process {PASS2_TEST_SIZE} games)" if PASS2_TEST_MODE else " OFF — full run"))
print(f"\n  Games with PASS 1 complete: {len(p2_candidates)} (of {len(games_to_generate)} total)")

if PASS2_MODEL_SELECTION == 'gemini-3' and not GEMINI_AVAILABLE:
    raise ValueError("Gemini-3 selected but module not available.")

# Resume state diagnostics
if Path(PASS2_OUTPUT_FILE).exists():
    _existing_p2 = pd.read_csv(PASS2_OUTPUT_FILE)
    _already_done_p2_count = len(_existing_p2)
    _already_done_p2_ids = set(_existing_p2['game_id'].tolist())
    _remaining_p2_count = len(p2_candidates[~p2_candidates['game_id'].isin(_already_done_p2_ids)])
    print(f"  Resuming — {_already_done_p2_count} games already done, {_remaining_p2_count} remaining.")
else:
    print(f"  No existing output — will run from scratch.")


PASS 2 Configuration:
  Model:       gpt-5.2
  Output file: Datasets/seeds/generated_r3_seeds_gpt5_2.csv
  Test mode:   True (will process 2 games)

  Games with PASS 1 complete: 5 (of 250 total)
  No existing output — will run from scratch.


In [73]:
# PASS 2 EXECUTION — with resume support and combined API+JSON+validation retry loop

_MAX_RETRIES = 5


def call_pass2_model(prompt: str, model: str) -> str:
    """
    Make a single API call and return raw content string.
    Raises the underlying exception on failure — retry logic is in the outer loop.
    """
    if model == 'gpt-5.2':
        response = get_completion_with_backoff(
            model='gpt-5.2',
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert in social deception games and deception theory. "
                               "You generate realistic game discussions with tactical communication patterns."
                },
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            max_completion_tokens=8192
        )
        return response.choices[0].message.content

    elif model == 'gemini-3':
        system_msg = ("You are an expert in social deception games and deception theory. "
                      "You generate realistic game discussions with tactical communication patterns.")
        full_prompt = f"{system_msg}\n\n{prompt}\n\nIMPORTANT: Return ONLY valid JSON as specified."
        response = get_gemini_completion(
            model='gemini-3.1-pro-preview',
            prompt=full_prompt,
            max_output_tokens=8192
        )
        content = response.choices[0].message.content
        # Strip markdown code fences if present
        for fence in ('```json', '```'):
            if content.startswith(fence):
                content = content[len(fence):]
        if content.endswith('```'):
            content = content[:-3]
        return content.strip()

    else:
        raise ValueError(f"Unknown model: {model}")


# ── Resume: load any already-completed games ─────────────────────────────────
already_done_p2_df = pd.DataFrame()
already_done_p2 = set()
if Path(PASS2_OUTPUT_FILE).exists():
    already_done_p2_df = pd.read_csv(PASS2_OUTPUT_FILE)
    already_done_p2 = set(already_done_p2_df['game_id'].tolist())

# Filter to games not yet done, then apply test/full limit
remaining_candidates = p2_candidates[~p2_candidates['game_id'].isin(already_done_p2)].copy().reset_index(drop=True)
if PASS2_TEST_MODE:
    remaining_candidates = remaining_candidates.iloc[:PASS2_TEST_SIZE].reset_index(drop=True)

if remaining_candidates.empty:
    print("✓ PASS 2 already complete — all games done. Skipping.")
else:
    pass2_results = []    # new results this run
    pass2_failed = []

    print(f"{'='*70}")
    print(f"PASS 2: Dialogue generation for {len(remaining_candidates)} games ({PASS2_MODEL_SELECTION})")
    if already_done_p2:
        print(f"  (Resuming — {len(already_done_p2)} games already done)")
    if PASS2_TEST_MODE:
        print(f"  ⚠  TEST MODE — processing {len(remaining_candidates)} games only")
    print(f"{'='*70}\n")

    for idx, game_row in remaining_candidates.iterrows():
        game_id = game_row['game_id']
        display_idx = idx + 1

        print(f"[{display_idx}/{len(remaining_candidates)}] {game_id}...", end=' ')

        pass1_matrix = pass1_matrix_lookup.get(game_id, {})
        if not pass1_matrix:
            print(f"✗ No PASS 1 matrix")
            pass2_failed.append((game_id, 'No PASS 1 matrix'))
            continue

        prompt = build_pass2_prompt(game_row, pass1_matrix)

        # ── Combined retry: API error → JSON parse error → validation failure ──
        _validated = None
        for _attempt in range(_MAX_RETRIES):
            # 1. API call (single attempt — raises on failure)
            try:
                raw_content = call_pass2_model(prompt, PASS2_MODEL_SELECTION)
            except Exception as _api_e:
                if _attempt < _MAX_RETRIES - 1:
                    _delay = 5 * (2 ** _attempt) + random.uniform(0, 2)
                    print(f"\n  API error (attempt {_attempt+1}/{_MAX_RETRIES}): {str(_api_e)[:80]}. Retrying in {_delay:.1f}s...", end=' ')
                    time.sleep(_delay)
                    continue
                else:
                    print(f"✗ API error after {_MAX_RETRIES} attempts: {_api_e}")
                    pass2_failed.append((game_id, f'API error: {_api_e}'))
                    break

            # 2. JSON parse — try direct, then repair_pass2_json
            try:
                generated = json.loads(raw_content)
                # Normalise double-encoded matrix_tactic_scale (common Gemini issue):
                # json.loads succeeded on the outer JSON, but the matrix value may still
                # be a JSON string rather than a parsed dict.
                _mts = generated.get('matrix_tactic_scale')
                if isinstance(_mts, str):
                    try:
                        generated['matrix_tactic_scale'] = json.loads(_mts)
                    except json.JSONDecodeError:
                        try:
                            generated['matrix_tactic_scale'] = json.loads(repair_matrix_json(_mts))
                        except json.JSONDecodeError:
                            pass  # validate_pass2_output will flag this as an error → retry
            except json.JSONDecodeError:
                try:
                    generated = repair_pass2_json(raw_content)
                except json.JSONDecodeError as _json_e:
                    if _attempt < _MAX_RETRIES - 1:
                        _delay = 5 * (2 ** _attempt) + random.uniform(0, 2)
                        print(f"\n  JSON parse error (attempt {_attempt+1}/{_MAX_RETRIES}): {str(_json_e)[:60]}. Retrying in {_delay:.1f}s...", end=' ')
                        time.sleep(_delay)
                        continue
                    else:
                        print(f"✗ JSON parse error after {_MAX_RETRIES} attempts: {_json_e}")
                        pass2_failed.append((game_id, f'JSON parse error: {_json_e}'))
                        break

            # 3. Validation
            is_valid, errors = validate_pass2_output(game_row, generated)
            if is_valid:
                _validated = generated
                break
            else:
                if _attempt < _MAX_RETRIES - 1:
                    _delay = 5 * (2 ** _attempt) + random.uniform(0, 2)
                    print(f"\n  Validation failed (attempt {_attempt+1}/{_MAX_RETRIES}): {errors[0]}. Retrying in {_delay:.1f}s...", end=' ')
                    time.sleep(_delay)
                else:
                    print(f"✗ Validation failed after {_MAX_RETRIES} attempts: {errors}")
                    pass2_failed.append((game_id, errors))

        if _validated is not None:
            print('✓')
            player_roles = json.loads(game_row['player_roles'])
            pass2_results.append({
                'game_id': game_id,
                'round_id': game_row['round_id'],
                'role_id': game_row['role_id'],
                'llm_alignment': player_roles.get(game_row['role_id'], ''),
                'player_roles': game_row['player_roles'],
                'public_history': game_row['public_history'],
                'prior_summary_gold': '',  # Populated post-hoc after all rounds complete
                'discussion_log': _validated['discussion_log'],
                'matrix_tactic_scale': format_matrix_tactic_scale(_validated['matrix_tactic_scale']),
            })

        # Incremental save every 10 games (union with already-done)
        if len(pass2_results) > 0 and len(pass2_results) % 10 == 0:
            combined_p2 = pd.concat(
                [already_done_p2_df, pd.DataFrame(pass2_results)],
                ignore_index=True
            )
            combined_p2.to_csv(PASS2_OUTPUT_FILE, index=False)
            print(f"  💾 Progress saved ({len(combined_p2)} total games in file)")

        time.sleep(1)

    # Final save (union with already-done)
    if pass2_results:
        combined_p2 = pd.concat(
            [already_done_p2_df, pd.DataFrame(pass2_results)],
            ignore_index=True
        )
        combined_p2.to_csv(PASS2_OUTPUT_FILE, index=False)

    print(f"\n{'='*70}")
    print(f"PASS 2 COMPLETE ({PASS2_MODEL_SELECTION})")
    print(f"  New successes this run: {len(pass2_results)} games")
    print(f"  Failed this run:        {len(pass2_failed)} games")
    total_in_file = len(already_done_p2) + len(pass2_results)
    print(f"  Total in output file:   {total_in_file} games")
    print(f"  Output: {PASS2_OUTPUT_FILE}")
    print(f"{'='*70}")

    if pass2_failed:
        print(f"\nFailed games:")
        for gid, err in pass2_failed:
            print(f"  {gid}: {err}")


PASS 2: Dialogue generation for 2 games (gpt-5.2)
  ⚠  TEST MODE — processing 2 games only

[1/2] G001... ✓
[2/2] G002... ✓

PASS 2 COMPLETE (gpt-5.2)
  New successes this run: 2 games
  Failed this run:        0 games
  Total in output file:   2 games
  Output: Datasets/seeds/generated_r3_seeds_gpt5_2.csv


## Step 10: Review Results

In [74]:
# Review PASS 2 output
if Path(PASS2_OUTPUT_FILE).exists():
    output_df = pd.read_csv(PASS2_OUTPUT_FILE)
    print(f"Generated {len(output_df)} rows in {PASS2_OUTPUT_FILE}")
    print(f"Columns: {output_df.columns.tolist()}")
    
    # Show first generated dialogue
    row = output_df.iloc[0]
    print(f"\n=== First Generated Game: {row['game_id']} ===")
    print(f"Protagonist (silent): {row['role_id']}")
    print(f"\nDiscussion Log:")
    print(row['discussion_log'])
    print(f"\nMatrix Tactic Scale:")
    print(row['matrix_tactic_scale'])
else:
    print("No output file found — run PASS 2 first.")

Generated 2 rows in Datasets/seeds/generated_r3_seeds_gpt5_2.csv
Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale']

=== First Generated Game: G001 ===
Protagonist (silent): P1

Discussion Log:
Discussion after Quest 3:

P4: "We can stop pretending the pattern is mysterious: P1 has now been on Quest 1 and Quest 3 and both failed. If you want two names that actually touch every fail in a clean way, it’s P1 and P3."

P3: "That’s a convenient story, but it ignores the real issue: every team with P1 has failed, and people kept waving it through anyway—unanimous just now. My two are P1 and P4, because P4’s been steering the blame every single time a result comes up."

P2: "Let’s pin the facts: Quest 1 was {P1,P4} and failed; Quest 2 was {P2,P3,P5} and failed; Quest 3 was {P1,P3} and failed—so P3 appears on two different failed teams with two different partners, which is the strongest o

## Next Steps

### Run both models:
1. **GPT-5.2**: Set `PASS2_MODEL_SELECTION = 'gpt-5.2'` → generates `generated_r3_seeds_gpt5_2.csv`
2. **Gemini-3**: Set `PASS2_MODEL_SELECTION = 'gemini-3'` → generates `generated_r3_seeds_gemini3.csv`

### Verification:
3. Run `log-gen-verifier.ipynb` on both PASS 2 outputs (same 5-criteria pipeline as R1)
   - Update input filenames to point to `generated_r3_seeds_*.csv`
   - Output: `verified_r3_seeds_combined.csv` + `verified_r3_criteria_scores.csv`

### Summary (`prior_summary_gold`):
- Generate summaries of Round 3 `summaries_r3.csv` after verification from `verified_r3_seeds_combined.csv`

### Merge
- Manually merge the combined csv `verified_r3_seeds_combined.csv` and generated summaries `summaries_r3.csv` into `Deception-Dataset.csv`